In [2]:
# =============================================================
# 05_full_pipeline_demo.ipynb
# Full Pipeline Demo — End-to-End Inference for a Single Store
# -------------------------------------------------------------
# Constructs one experimental store by randomly sampling
# products from the REAL dataset, then runs all 4 stages.
#
# Input  : 네이버_실험용샘플_1dep.csv   (raw data)
#          models/best_tfidf_model.pkl
#          data/label_encoder.pkl
# Output : results/pipeline_demo_output.json
#          results/pipeline_demo_store_products.csv
# =============================================================

# ── Cell 1: Imports ──────────────────────────────────────────
import pickle, json, os
import numpy as np
import pandas as pd
from collections import Counter
from scipy.stats import entropy as scipy_entropy

os.makedirs('results', exist_ok=True)
print("Libraries loaded.")


# ── Cell 2: API Key Setup ────────────────────────────────────
import os
os.environ['OPENAI_API_KEY'] = ''
print("API key set.")


# ── Cell 3: Load Raw Data & Model ────────────────────────────
df_raw = pd.read_csv('네이버_실험용샘플_1dep.csv', encoding='utf-8')
# df_raw = pd.read_csv('네이버_실험용샘플_1dep.csv', encoding='cp949')

with open('data/label_encoder.pkl', 'rb') as f:
    le = pickle.load(f)

with open('models/best_tfidf_model.pkl', 'rb') as f:
    saved = pickle.load(f)
vec, clf = saved['vectorizer'], saved['model']

NUM_CLASSES = len(le.classes_)
print(f"Raw data loaded : {len(df_raw):,} records")
print(f"Classes         : {le.classes_.tolist()}")


# ── Cell 3: Construct Experimental Store from Real Data ──────
"""
Experimental store construction:
  - Assign a fictional store ID (STORE_EXP_001)
  - Randomly sample N_PRODS products from the raw dataset
  - Intentionally sample from mixed categories to reflect
    a realistic multi-category seller scenario
  - Base date taken from the dataset collection period
"""
STORE_ID    = "STORE_EXP_001"
STORE_NAME  = "Experimental Store #001"
BASE_DATE   = "2023-11-01"
N_PRODS     = 30      # number of products in this experimental store
RANDOM_SEED = 2024

# Sample N_PRODS products randomly from the full raw dataset
store_products = df_raw.sample(n=N_PRODS, random_state=RANDOM_SEED).reset_index(drop=True)

# Keep only relevant columns
store_products = store_products[
    ['prodId', 'prodName', 'naver_catName_depth1']
].rename(columns={
    'prodId'              : 'prod_id',
    'prodName'            : 'prod_name',
    'naver_catName_depth1': 'true_category'
})

print(f"\nExperimental Store : {STORE_NAME}  (ID: {STORE_ID})")
print(f"Base Date          : {BASE_DATE}")
print(f"Total Products     : {len(store_products)}")
print(f"\nTrue category distribution (ground truth):")
print(store_products['true_category'].value_counts().to_string())
print(f"\nProduct list (sample 10):")
print(store_products[['prod_name','true_category']].head(10).to_string(index=False))

# Save store product list
store_products.to_csv(
    'results/pipeline_demo_store_products.csv',
    index=False, encoding='utf-8-sig'
)
print("\nSaved → results/pipeline_demo_store_products.csv")


# ── Cell 4: Stage 1 — Product-Level Classification ──────────
print("\n" + "─" * 55)
print("STAGE 1: Product-Level Classification")
print("─" * 55)

import re

def clean_text(text):
    text = str(text)
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'[^\w\s가-힣a-zA-Z0-9]', ' ', text)
    return text.strip()

prod_names_clean = store_products['prod_name'].apply(clean_text).values

X_store      = vec.transform(prod_names_clean)
pred_labels  = clf.predict(X_store).astype(int)
pred_probs   = clf.predict_proba(X_store)
pred_cats    = le.inverse_transform(pred_labels)
confidences  = pred_probs.max(axis=1)

stage1_df = store_products.copy()
stage1_df['pred_category'] = pred_cats
stage1_df['confidence']    = confidences.round(4)
stage1_df['correct']       = (stage1_df['pred_category'] == stage1_df['true_category'])

print(f"\nPer-product classification results:")
print(stage1_df[['prod_name','true_category','pred_category','confidence','correct']]
      .to_string(index=False))

prod_acc = stage1_df['correct'].mean()
print(f"\nProduct-level accuracy (this store): {prod_acc:.4f}")


# ── Cell 5: Stage 2 — Store-Level Aggregation & SCS ─────────
print("\n" + "─" * 55)
print("STAGE 2: Store-Level Aggregation & SCS")
print("─" * 55)

def compute_scs(pred_labels_arr, n_classes):
    """
    SCS = (n_dominant / N) × 1 / (1 + H)
    H : Shannon Entropy of predicted category distribution
    """
    n      = len(pred_labels_arr)
    counts = np.bincount(pred_labels_arr, minlength=n_classes)
    n_dom  = counts.max()
    p      = counts / n
    H      = scipy_entropy(p + 1e-10, base=2)
    return float((n_dom / n) * (1.0 / (1.0 + H)))

def majority_vote(pred_labels_arr):
    return Counter(pred_labels_arr).most_common(1)[0][0]

def confidence_weighted_vote(pred_labels_arr, pred_probs_arr):
    return int(np.argmax(pred_probs_arr.sum(axis=0)))

seg_mv  = majority_vote(pred_labels)
seg_cwv = confidence_weighted_vote(pred_labels, pred_probs)
scs_val = compute_scs(pred_labels, NUM_CLASSES)

cat_dist = Counter(pred_cats)
n_total  = len(pred_labels)

print(f"\nPredicted category distribution:")
for cat, cnt in sorted(cat_dist.items(), key=lambda x: -x[1]):
    bar = "█" * int(cnt / n_total * 30)
    print(f"  {cat:<20} {cnt:>3}건 ({cnt/n_total*100:5.1f}%)  {bar}")

print(f"\nMajority Voting     → {le.inverse_transform([seg_mv])[0]}")
print(f"Confidence-Weighted → {le.inverse_transform([seg_cwv])[0]}")
print(f"SCS                 : {scs_val:.4f}")

TAU = 0.30   # update with optimal τ from 03_stage2 experiment
flag = "AUTO" if scs_val >= TAU else "MANUAL_REVIEW"
print(f"Flag (τ={TAU})       : {flag}")

assigned_seg = le.inverse_transform([seg_cwv])[0]


# ── Cell 6: Stage 3 — Credit Model Assignment ───────────────
print("\n" + "─" * 55)
print("STAGE 3: Credit Model Assignment")
print("─" * 55)

# Assumption A1: Segment-specific alternative credit models exist
SEG_TO_MODEL = {cat: f"{cat} 전문 대안신용평가 모형" for cat in le.classes_}
assigned_model = SEG_TO_MODEL[assigned_seg]

# Ground truth: most frequent TRUE category in this store
true_dominant = store_products['true_category'].value_counts().idxmax()
seg_correct   = (assigned_seg == true_dominant)

print(f"Assigned Seg      : {assigned_seg}")
print(f"True dominant cat : {true_dominant}")
print(f"Seg assignment    : {'✓ Correct' if seg_correct else '✗ Incorrect'}")
print(f"Assigned Model    : {assigned_model}")
print(f"[Assumption A1]   : Segment-specific models assumed to exist.")


# ── Cell 7: Stage 4 — LLM Notice Generation ─────────────────
print("\n" + "─" * 55)
print("STAGE 4: LLM-Based Notice Generation")
print("─" * 55)

SYSTEM_PROMPT = """You are an AI assistant for a Korean alternative credit evaluation firm.
Generate a formal credit evaluation basis notice in Korean.
Include: base date, product count, category distribution (top 3),
assigned segment, credit model name, and objection instructions.
Output ONLY the notice text. 150–250 Korean characters."""

dist_lines = "\n".join(
    f"  - {cat}: {cnt}건 ({cnt/n_total*100:.1f}%)"
    for cat, cnt in sorted(cat_dist.items(), key=lambda x: -x[1])[:5]
)
user_prompt = f"""다음 정보를 바탕으로 신용평가 근거 안내문을 작성하세요.

- 스토어 ID: {STORE_ID}
- 평가 기준일: {BASE_DATE}
- 분석 상품 수: {n_total}건
- 상품 카테고리 분포 (예측 기반):
{dist_lines}
- 배정된 업종 세그먼트: {assigned_seg}
- 적용 신용평가 모형: {assigned_model}
- SCS(세그먼트 신뢰도): {scs_val:.3f}
- 배정 방식: {'자동 배정' if flag == 'AUTO' else '수동 검토 후 배정'}"""

def call_llm(user_prompt, system_prompt=SYSTEM_PROMPT):
    try:
        from openai import OpenAI
        client = OpenAI()   # reads OPENAI_API_KEY from env
        resp = client.chat.completions.create(
            model="gpt-4o-mini",
            max_tokens=512,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user",   "content": user_prompt}
            ]
        )
        return resp.choices[0].message.content.strip()
    except ImportError:
        return "[openai not installed — pip install openai]"
    except Exception as e:
        return f"[API Error: {e}]"

notice = call_llm(user_prompt)
print(f"\n{'='*55}")
print("GENERATED NOTICE:")
print('='*55)
print(notice)
print('='*55)


# ── Cell 8: Save Full Output ─────────────────────────────────
output = {
    "store_id"   : STORE_ID,
    "store_name" : STORE_NAME,
    "base_date"  : BASE_DATE,
    "n_products" : n_total,
    "data_source": "Naver SmartStore crawled dataset (2023-11)",
    "random_seed": RANDOM_SEED,
    "stage1": {
        "model"            : "TF-IDF + LR",
        "product_accuracy" : round(prod_acc, 4),
        "per_product"      : stage1_df[
            ['prod_id','prod_name','true_category',
             'pred_category','confidence','correct']
        ].to_dict('records')
    },
    "stage2": {
        "predicted_distribution": {
            k: {"count": v, "ratio": round(v/n_total, 4)}
            for k, v in cat_dist.items()
        },
        "majority_voting_seg"    : le.inverse_transform([seg_mv])[0],
        "confidence_weighted_seg": assigned_seg,
        "scs"                    : round(scs_val, 6),
        "tau"                    : TAU,
        "flag"                   : flag
    },
    "stage3": {
        "assigned_seg"   : assigned_seg,
        "true_dominant"  : true_dominant,
        "seg_correct"    : seg_correct,
        "assigned_model" : assigned_model,
        "assumption_A1"  : "Segment-specific credit models assumed to exist"
    },
    "stage4": {
        "notice_text"  : notice,
        "notice_length": len(notice),
        "llm_model"    : "gpt-4o-mini"
    }
}

with open('results/pipeline_demo_output.json', 'w', encoding='utf-8') as f:
    json.dump(output, f, ensure_ascii=False, indent=2)
print("\nSaved → results/pipeline_demo_output.json")


# ── Cell 9: Pipeline Summary ─────────────────────────────────
print("\n" + "=" * 55)
print("FULL PIPELINE SUMMARY")
print("=" * 55)
print(f"  Store ID          : {STORE_ID}")
print(f"  Products analyzed : {n_total}")
print(f"  Product-level Acc : {prod_acc:.4f}")
print(f"  Assigned Seg      : {assigned_seg}")
print(f"  True dominant cat : {true_dominant}")
print(f"  Seg correct       : {'✓' if seg_correct else '✗'}")
print(f"  SCS               : {scs_val:.4f}  (Flag: {flag})")
print(f"  Credit Model      : {assigned_model}")
print(f"  Notice length     : {len(notice)} chars")
print("=" * 55)
print("Pipeline complete.")

Libraries loaded.
API key set.
Raw data loaded : 110,000 records
Classes         : ['가구/인테리어', '도서', '디지털/가전', '생활/건강', '스포츠/레저', '식품', '여가/생활편의', '출산/육아', '패션의류', '패션잡화', '화장품/미용']

Experimental Store : Experimental Store #001  (ID: STORE_EXP_001)
Base Date          : 2023-11-01
Total Products     : 30

True category distribution (ground truth):
도서         5
여가/생활편의    4
생활/건강      4
디지털/가전     4
스포츠/레저     3
출산/육아      3
가구/인테리어    2
화장품/미용     2
패션잡화       1
식품         1
패션의류       1

Product list (sample 10):
                                                       prod_name true_category
                                               보성아산병원근조화환 배달비 무료       여가/생활편의
                                  휴대용 차량용 초기 진압용 간이 소화기 미니소화기 중형         생활/건강
                        사무실 휴게실 업소용 5방향 파워난방 전기히터 중형급 온열기 바퀴형 난로        디지털/가전
                                     포토 박스 화이트 색상 리필 20장 프레임 사진북         생활/건강
                          식자재 보스코도로 블랙트러플 페이스트 1 180g 식당 업소용 식재료       여가/생활편의
       